# Sudoku solver

Fill the following 9-by-9 grid, keeping all the given numbers. A dash marks an unknown entry.

| Row | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 1 | 8 | — | — | — | — | — | — | — | — |
| 2 | — | — | 3 | 6 | — | — | — | — | — |
| 3 | — | 7 | — | — | 9 | — | 2 | — | — |
| 4 | — | 5 | — | — | — | 7 | — | — | — |
| 5 | — | — | — | — | 4 | 5 | 7 | — | — |
| 6 | — | — | — | 1 | — | — | — | 3 | — |
| 7 | — | — | 1 | — | — | — | — | 6 | 8 |
| 8 | — | — | 8 | 5 | — | — | — | 1 | — |
| 9 | — | 9 | — | — | — | — | 4 | — | — |

- Each row must contain each number from 1 to 9 exactly once.
- Each column must contain each number from 1 to 9 exactly once.
- Each of the nine 3-by-3 blocks must contain each number from 1 to 9 exactly once.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. All required packages are included; no external data files are needed.

#### Problem data

In [ ]:
# Given data. Unknown entries are specified as 0.
given = [
  8 0 0 0 0 0 0 0 0
  0 0 3 6 0 0 0 0 0
  0 7 0 0 9 0 2 0 0
  0 5 0 0 0 7 0 0 0
  0 0 0 0 4 5 7 0 0
  0 0 0 1 0 0 0 3 0
  0 0 1 0 0 0 0 6 8
  0 0 8 5 0 0 0 1 0
  0 9 0 0 0 0 4 0 0
];

In [ ]:
# Helper function to print a Sudoku grid.
function printSudoku(arr)
    println("+-------+-------+-------+")
    for p in 1:3:9
        for q in 0:2
            print("| ")
            for r in 1:3:9
                for s in 0:2
                    digit = round(Int, arr[p+q,r+s])
                    digit == 0 ? print(" ") : print(digit)
                    print(" ")
                end
                print("| ")
            end
            println()
        end
        println("+-------+-------+-------+")
    end
end

In [ ]:
printSudoku(given)

### Binary feasibility model

Let $x_{ijk}=1$ when row $i$, column $j$ contains digit $k$, and 0 otherwise.
The constraints assign one digit to each cell and require each digit once in
every row, column, and 3-by-3 block. A given digit fixes its corresponding
binary variable to 1. There is no objective: any feasible completion solves
the puzzle.

In [ ]:
using JuMP, HiGHS

m = Model(HiGHS.Optimizer)
set_silent(m)

@variable(m, x[1:9,1:9,1:9], Bin)

# exactly one number per cell
for i in 1:9
  for j in 1:9
    @constraint(m, sum(x[i,j,k] for k in 1:9) == 1)
  end
end

# exactly one of each number per row
for i in 1:9
  for k in 1:9
    @constraint(m, sum(x[i,j,k] for j in 1:9) == 1)
  end
end

# exactly one of each number per column
for j in 1:9
  for k in 1:9
    @constraint(m, sum(x[i,j,k] for i in 1:9) == 1)
  end
end

# exactly one of each number per 3x3 block
for k in 1:9
  for p in 0:3:6
    for q in 0:3:6
      @constraint(m, sum(x[p+i,q+j,k] for i in 1:3, j in 1:3) == 1)
    end
  end
end

# initial conditions
for i in 1:9
  for j in 1:9
    if given[i,j] != 0
      @constraint(m, x[i,j,given[i,j]] == 1)
    end
  end
end

@time(optimize!(m))

if !is_solved_and_feasible(m; allow_local = false)
    error("No optimal primal solution: " *
        "$(termination_status(m)), $(primal_status(m))")
end

# Binary values may differ slightly from 0 or 1 within solver tolerances.
solution = [sum(k * (value(x[i,j,k]) > 0.5) for k in 1:9)
            for i in 1:9, j in 1:9]
println("The given problem is:")
printSudoku(given)
println("The solution is:")
printSudoku(solution)